# Manna adult difficulty reallocation

## tl;dr

This notebook independently reconciles the complete 40,020-question adult reallocation. It verifies the 9,791 changes, the final 35 → 58 → 80 median progression, the 20-question minimum category-tier pool, and zero drift between the shipping labels and the frozen allocation manifest.

## Context & Methods

The adult difficulty tiers are Layperson, Deacon, and Pastor. A deterministic 0–100 content score measures recall obscurity, task complexity, specificity, distractor competitiveness, and language burden. Task-family guardrails override raw score bands for specialist work such as original-language vocabulary, Church Father attribution, exact cross-references, and academic biblical studies.

### Key assumptions

- Records without `ageGroup` are adult questions; child records are out of scope.
- `before` is reconstructed from calibrated source labels, so the baseline is stable across reruns.
- `final` includes documented operational-floor adjustments needed to preserve at least 20 questions in every adult category-tier pool.
- Content scores are editorial estimates, not empirical player-performance measurements.

In [ ]:
from pathlib import Path
import csv
import json
import statistics

root = Path.cwd()
if not (root / 'reports/difficulty_reallocation').exists():
    raise RuntimeError('Run this notebook from the Manna repository root')

reallocation_dir = root / 'reports/difficulty_reallocation'
audit_dir = root / 'reports/difficulty_audit'
summary = json.loads((reallocation_dir / 'summary.json').read_text(encoding='utf-8'))
audit = json.loads((audit_dir / 'summary.json').read_text(encoding='utf-8'))
overrides = json.loads((root / 'difficulty_overrides.json').read_text(encoding='utf-8'))
with (reallocation_dir / 'all_questions.csv').open(encoding='utf-8', newline='') as handle:
    all_rows = list(csv.DictReader(handle))
with (reallocation_dir / 'changes.csv').open(encoding='utf-8', newline='') as handle:
    changed_rows = list(csv.DictReader(handle))
print(f"Loaded {len(all_rows):,} adult rows and {len(changed_rows):,} changed rows.")

## Data

The question-level CSV is the complete reviewed population, not a sample. The change CSV is its off-diagonal subset. The summary contains derived distributions, transitions, family mix, and category pools. The override manifest is the build-time source of truth keyed by stable question ID.

In [ ]:
# Reconcile the four persisted outputs before calculating findings.
assert len(all_rows) == summary['adult_question_count'] == 40_020
assert len(changed_rows) == summary['changed_question_count'] == 9_791
assert len(overrides['overrides']) == len(changed_rows)
assert len({row['id'] for row in all_rows}) == len(all_rows)
assert len({row['id'] for row in changed_rows}) == len(changed_rows)
assert {row['id'] for row in changed_rows} == {row['id'] for row in overrides['overrides']}
print('Population, change log, and frozen manifest reconcile exactly.')

## Results

First, recompute tier counts and score distributions directly from the complete question-level file. This avoids trusting pre-aggregated summary values.

In [ ]:
tier_order = ['Layperson', 'Deacon', 'Pastor']
def percentile(values, fraction):
    values = sorted(values)
    position = (len(values) - 1) * fraction
    lower = int(position)
    upper = min(lower + 1, len(values) - 1)
    weight = position - lower
    return values[lower] * (1 - weight) + values[upper] * weight

recomputed = []
for tier in tier_order:
    scores = [int(row['score']) for row in all_rows if row['final'] == tier]
    recomputed.append({
        'tier': tier,
        'count': len(scores),
        'median': statistics.median(scores),
        'p25': percentile(scores, 0.25),
        'p75': percentile(scores, 0.75),
        'min': min(scores),
        'max': max(scores),
    })
assert recomputed == summary['after']
for row in recomputed:
    print(f"{row['tier']:10} {row['count']:>6,}  median {row['median']:>4}  IQR {row['p25']:.0f}–{row['p75']:.0f}")

In [ ]:
# Recompute all nine transitions and the before/after separation.
transition_counts = {}
for row in all_rows:
    key = (row['before'], row['final'])
    transition_counts[key] = transition_counts.get(key, 0) + 1
expected_transitions = {(row['before'], row['after']): row['count'] for row in summary['transitions']}
assert transition_counts == expected_transitions
unchanged = sum(count for (before, after), count in transition_counts.items() if before == after)
before_medians = [row['median'] for row in summary['before']]
after_medians = [row['median'] for row in summary['after']]
before_gap = min(b - a for a, b in zip(before_medians, before_medians[1:]))
after_gap = min(b - a for a, b in zip(after_medians, after_medians[1:]))
assert after_gap == summary['minimum_adjacent_median_gap_after'] == 22
print(f"Unchanged: {unchanged:,} ({unchanged / len(all_rows):.1%})")
print(f"Reallocated: {len(changed_rows):,} ({len(changed_rows) / len(all_rows):.2%})")
print(f"Minimum adjacent median gap: {before_gap:.0f} → {after_gap:.0f}")

In [ ]:
# Verify build-operability and final allocation integrity.
category_counts = {}
for row in all_rows:
    key = (row['category'], row['final'])
    category_counts[key] = category_counts.get(key, 0) + 1
categories = sorted({row['category'] for row in all_rows})
missing_cells = [(category, tier) for category in categories for tier in tier_order if (category, tier) not in category_counts]
minimum_pool = min(category_counts.values())
assert not missing_cells
assert minimum_pool == summary['minimum_category_tier_pool'] == audit['minimum_category_tier_pool'] == 20
assert audit['mismatch_count'] == 0
assert audit['allocation_drift_count'] == 0
print(f"Category-tier cells: {len(category_counts)} across {len(categories)} categories")
print(f"Smallest pool: {minimum_pool}; manifest mismatch: 0; allocation drift: 0")

## Takeaways

- The final labels have a clear population-level progression: median content scores are 35, 58, and 80.
- The change is substantial but selective: 24.47% moved and 75.5% stayed in place.
- Every category supports the app's maximum 20-question adult game length in every tier.
- The shipping labels exactly match the frozen allocation, so future builds can reject stale or orphaned decisions.
- Individual labels remain content-based hypotheses. Player response data should eventually provide the empirical calibration layer.